In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [ ]:
# !pip install -qU wandb

In [ ]:
# from google.colab import userdata


Okay, here's the Markdown for that code snippet:

## 2. Weights & Biases (W&B) Setup

This cell initializes the connection to **Weights & Biases (W&B)**. W&B is a popular platform for experiment tracking, dataset versioning, and model management in machine learning projects.

*   **`import wandb`**: This line imports the `wandb` library.
*   **`wandb.login(key=userdata.get('wandb'))`**: This line logs into your W&B account.
    *   It's crucial for tracking the fine-tuning process, allowing you to monitor metrics like loss, learning rate, and potentially evaluation scores in real-time through the W&B dashboard.
    *   `userdata.get('wandb')` is used to securely access your W&B API key. This is a good practice, especially in environments like Google Colab or Kaggle, where you can store secret keys without hardcoding them directly into the notebook. You would typically set this key in the "Secrets" manager of your notebook environment.

By logging into W&B, all subsequent training runs initiated with W&B integration will automatically send their logs, metrics, and model checkpoints (if configured) to your W&B project workspace. This helps in comparing different experiments, visualizing results, and collaborating with others.


In [ ]:
# import wandb

In [ ]:
# wandb.login(key=userdata.get('wandb'))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: amro-eidd (amro-eidd-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

### Unsloth

Okay, here's the Markdown for this code block:

## 3. Model and Tokenizer Initialization with Unsloth

This cell focuses on loading the pre-trained large language model (LLM) and its corresponding tokenizer. We're leveraging Unsloth's `FastLanguageModel` for optimized loading and memory efficiency, particularly when using 4-bit quantized models.

**Key Steps and Parameters:**

1.  **Import necessary libraries**:
    *   `FastLanguageModel` from `unsloth`: This is Unsloth's optimized class for loading and working with LLMs.
    *   `torch`: The PyTorch library, which is the underlying framework for many LLMs.

2.  **Configuration Parameters**:
    *   `max_seq_length = 2048`: This defines the maximum number of tokens the model can process in a single input sequence. Unsloth handles RoPE (Rotary Positional Embedding) scaling internally, allowing flexibility in this choice. For essay grading, a sufficiently large `max_seq_length` is important to accommodate the question, reference answer, student answer, and mark scheme.
    *   `dtype = None`: This setting allows Unsloth to automatically detect the optimal data type (e.g., `float16` for Tesla T4/V100 GPUs, `bfloat16` for Ampere+ GPUs) for the model based on the available hardware. This can improve performance and reduce memory usage.
    *   `load_in_4bit = True`: This crucial parameter instructs Unsloth to load the model using 4-bit quantization. Quantization reduces the model's precision (and thus its size and memory footprint) with minimal impact on performance for many tasks. This makes it feasible to run larger models on consumer-grade hardware.

3.  **`fourbit_models` List (Informational)**:
    *   This list showcases a variety of 4-bit quantized models available directly through Unsloth. These models are pre-quantized, leading to significantly faster download times and reduced risk of Out-Of-Memory (OOM) errors. The project specifies using a Mistral-7B variant.

4.  **Loading the Model and Tokenizer**:
    *   `model, tokenizer = FastLanguageModel.from_pretrained(...)`: This is the core command that loads the model and tokenizer.
        *   `model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit"`: This specifies the exact model to be loaded from the Hugging Face Hub. We are using a 4-bit quantized version of Mistral-7B Instruct v0.3 provided by Unsloth, which is suitable for instruction-following tasks like essay grading.
        *   The `max_seq_length`, `dtype`, and `load_in_4bit` parameters defined earlier are passed here.
        *   The `token` argument (commented out) would be used if loading a "gated" model from Hugging Face that requires authentication (e.g., some Llama models).

**Output:**

*   `model`: The loaded 4-bit quantized Mistral-7B instruction-tuned model, ready for fine-tuning or inference.
*   `tokenizer`: The tokenizer associated with the Mistral model, responsible for converting text data into a format the model can understand (tokens) and vice-versa.

By using Unsloth and 4-bit quantization, we aim for faster training and reduced memory consumption, which is especially beneficial for this project.


In [2]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Meta-Llama-3.1-8B-bnb-4bit",      # Llama-3.1 15 trillion tokens model 2x faster!
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Meta-Llama-3.1-70B-bnb-4bit",
    "unsloth/Meta-Llama-3.1-405B-bnb-4bit",    # We also uploaded 4bit for 405b!
    "unsloth/Mistral-Nemo-Base-2407-bnb-4bit", # New Mistral 12b 2x faster!
    "unsloth/Mistral-Nemo-Instruct-2407-bnb-4bit",
    "unsloth/mistral-7b-v0.3-bnb-4bit",        # Mistral v3 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.6.1: Fast Mistral patching. Transformers: 4.52.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/141k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [5]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Grade the student's answer to the essay question based on the reference answer and the provided mark scheme. Give a score 0-4 and rationale.",  # instruction
        """Question: What is the role of attention mechanisms in transformer models?

          Reference Answer: Attention mechanisms allow transformer models to weigh the importance of different words in a sequence when making predictions. They help the model focus on relevant parts of the input, regardless of their position, enabling it to capture context and relationships effectively. This is critical for handling long-range dependencies in language.

          Student Answer: Attention tells the model which words matter more. It helps with understanding context even when words are far apart.

          Mark Scheme:
          1. Describes attention as weighing the importance of different words.
          2. Mentions the ability to focus on relevant parts of input.
          3. Explains how attention captures context or relationships.
          4. Refers to handling long-range dependencies or position-independence.""",  # input
        "",  # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

<s> Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Grade the student's answer to the essay question based on the reference answer and the provided mark scheme. Give a score 0-4 and rationale.

### Input:
Question: What is the role of attention mechanisms in transformer models?

          Reference Answer: Attention mechanisms allow transformer models to weigh the importance of different words in a sequence when making predictions. They help the model focus on relevant parts of the input, regardless of their position, enabling it to capture context and relationships effectively. This is critical for handling long-range dependencies in language.

          Student Answer: Attention tells the model which words matter more. It helps with understanding context even when words are far apart.

          Mark Scheme:
          1. Describes attention as weighing the i

In [7]:
import pandas as pd
from transformers import TextStreamer
import torch

# Load the CSV
df = pd.read_csv("science_qa_dataset_test.csv")
submission_df = pd.read_csv("science_qa_submission.csv")

In [13]:
# Prompt template
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""



In [14]:
# Setup model
FastLanguageModel.for_inference(model)  # Assuming 'model' and 'tokenizer' are already loaded
text_streamer = TextStreamer(tokenizer)

In [15]:
# Loop over each row in the dataset
for idx, row in df.iterrows():
    instruction = "Grade the student's answer to the essay question based on the reference answer and the provided mark scheme. Give a score 0-4 in format Score: and rationale."

    # Build the input using question, reference answer, student answer, and mark schemes
    input_text = f"""Question: {row['question']}

Reference Answer: {row['reference_answer']}

Student Answer: {row['student_answer']}

Mark Scheme:
1. {row['mark_scheme_1']}
2. {row['mark_scheme_2']}
3. {row['mark_scheme_3']}
4. {row['mark_scheme_4']}"""

    # Build the full prompt
    prompt = alpaca_prompt.format(instruction, input_text, "")

    # Tokenize and move to CUDA
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    # Generate model output
    outputs = model.generate(**inputs, streamer=None, max_new_tokens=128)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract score (look for first digit between 0–4 in output)
    import re
    match = re.search(r"Score:\s*([0-4])", generated_text)
    score = int(match.group(1)) if match else None

    print(generated_text)
    print(f"the id is {idx}",score)
    # Update the submission DataFrame
    submission_df.at[idx, "score_before_tuning"] = score

# Save the updated submission file
submission_df.to_csv("science_qa_submission.csv", index=False)


Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Grade the student's answer to the essay question based on the reference answer and the provided mark scheme. Give a score 0-4 in format Score: and rationale.

### Input:
Question: What is the main role of Proteins in Human Body?

Reference Answer: Building blocks of cells; essential for cell structure, enzymes, and signaling.

Student Answer: it is used to produce cells; essential for cell structure

Mark Scheme:
1. mention the production of cells
2. mention enzymes
3. Uses term 'cell structure'
4. Uses term 'signaling'

### Response:
Score: 2

Rationale: The student's answer correctly mentions that proteins are used to produce cells and are essential for cell structure, which aligns with the first two points in the mark scheme. However, the student does not mention enzymes or signaling, which are also important 

KeyboardInterrupt: 

<a name="Save"></a>
## Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
# model.save_pretrained("lora_model")  # Local saving
# tokenizer.save_pretrained("lora_model")
# # model.push_to_hub("your_name/lora_model", token = "...") # Online saving
# # tokenizer.push_to_hub("your_name/lora_model", token = "...") # Online saving

('lora_model/tokenizer_config.json',
 'lora_model/special_tokens_map.json',
 'lora_model/chat_template.jinja',
 'lora_model/tokenizer.model',
 'lora_model/added_tokens.json',
 'lora_model/tokenizer.json')

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
# if True:
#     from unsloth import FastLanguageModel
#     model, tokenizer = FastLanguageModel.from_pretrained(
#         model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
#         max_seq_length = max_seq_length,
#         dtype = dtype,
#         load_in_4bit = load_in_4bit,
#     )
#     FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# # alpaca_prompt = You MUST copy from above!

# inputs = tokenizer(
# [
#     alpaca_prompt.format(
#         "Grade the student's answer to the essay question based on the reference answer and the provided mark scheme. Give a score and rationale.",  # instruction
#         """Question: What is the role of attention mechanisms in transformer models?

#           Reference Answer: Attention mechanisms allow transformer models to weigh the importance of different words in a sequence when making predictions. They help the model focus on relevant parts of the input, regardless of their position, enabling it to capture context and relationships effectively. This is critical for handling long-range dependencies in language.

#           Student Answer: Attention tells the model which words matter more. It helps with understanding context even when words are far apart.

#           Mark Scheme:
#           1. Describes attention as weighing the importance of different words.
#           2. Mentions the ability to focus on relevant parts of input.
#           3. Explains how attention captures context or relationships.
#           4. Refers to handling long-range dependencies or position-independence.""",  # input
#         "",  # output - leave this blank for generation!
#     )
# ], return_tensors = "pt").to("cuda")

# from transformers import TextStreamer
# text_streamer = TextStreamer(tokenizer)
# _ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

==((====))==  Unsloth 2025.5.8: Fast Mistral patching. Transformers: 4.52.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Will load lora_model as a legacy tokenizer.


<s>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Grade the student's answer to the essay question based on the reference answer and the provided mark scheme. Give a score and rationale.

### Input:
Question: What is the role of attention mechanisms in transformer models?

          Reference Answer: Attention mechanisms allow transformer models to weigh the importance of different words in a sequence when making predictions. They help the model focus on relevant parts of the input, regardless of their position, enabling it to capture context and relationships effectively. This is critical for handling long-range dependencies in language.

          Student Answer: Attention tells the model which words matter more. It helps with understanding context even when words are far apart.

          Mark Scheme:
          1. Describes attention as weighing the import

You can also use Hugging Face's `AutoModelForPeftCausalLM`. Only use this if you do not have `unsloth` installed. It can be hopelessly slow, since `4bit` model downloading is not supported, and Unsloth's **inference is 2x faster**.